# Word2Vec

## Train a Word2Vec Model with all the words from Keywords, Genres and Overview

In [ ]:
!pip install gensim

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 63.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 14.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.14.1
    Uninstalling scipy-1.14.1:
      Successfully uninstalled scipy-1.14.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
!pip install --force-reinstall numpy==1.24.4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 76.4 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.5
    Uninstalling numpy-2.2.5:
      Successfully uninstalled numpy-2.2.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
blis 1.0.2 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.24.4 which is incompatible.
thinc 9.1.1 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.24.4 which is incompatible.
spacy 3.8.5 requires thinc<8.4.0,>=8.3.4, but you have thinc 9.1.1 which is incompatible.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
blosc2 3.3.0 requires numpy>=1.26, but you have numpy 1.24.4 which is incompatible.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.24.4 which is inc

In [ ]:
import pandas as pd

df = pd.read_csv('../combined.csv')
df.fillna('',inplace=True)

In [ ]:
from gensim.models import Word2Vec

sentences = []
for index, row in df.iterrows():
    # Tokenize and add keywords, genres, and overview to the sentences list
    sentences.extend([row['keywords'].lower().split()])
    sentences.extend([row['genres'].lower().split()])
    sentences.extend([row['overview'].lower().split()])

# Train the Word2Vec model
model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)

# Save the trained model
model.save("custom_word2vec.model")

# Load the trained model (if needed)
model = Word2Vec.load("custom_word2vec.model")

## From the trained Word2Vec model extract the vectr representation of the words of each datapoint

In [ ]:
import numpy as np
from gensim.models import Word2Vec

model = Word2Vec.load("custom_word2vec.model")

def get_vector_representation(text, method='average'):
    """
    Calculates the vector representation of a text.

    Args:
        text: The input text (string).
        method: The method for combining word vectors ('average' or 'sum').

    Returns:
        The vector representation as a NumPy array.
    """
    words = text.lower().split()
    vectors = [model.wv[word] for word in words if word in model.wv]
    if vectors:
        if method == 'average':
            return np.mean(vectors, axis=0)
        elif method == 'sum':
            return np.sum(vectors, axis=0)
    return np.zeros(model.vector_size)  # Return zero vector if no words are found

# Apply the function to the relevant columns
df['keywords_vector'] = df['keywords'].apply(lambda x: get_vector_representation(x, method='average'))
df['genres_vector'] = df['genres'].apply(lambda x: get_vector_representation(x, method='average'))
df['overview_vector'] = df['overview'].apply(lambda x: get_vector_representation(x, method='sum'))

# Now you have new columns in your DataFrame containing the vector representations.

In [ ]:
len(df['keywords_vector'][0])

100

In [ ]:
df.to_csv('combined_with_vectors.csv', index=False)